# Mirror 7 — V5 Native GPU Training and Knowledge-Transfer Pipeline

This notebook runs the native V5 model independently of any teacher at inference time. Teacher demonstrations are optional training data, not a production dependency.

### Pipeline Highlights
- **Hardware**: CUDA detection with FP16/BF16 mixed precision
- **Architecture**: BiGRU encoder, multi-head cross-attention, recurrent-context gate, output gate, embedding dropout, scheduled sampling (enhanced default: 974,019 parameters)
- **Efficiency**: dynamic bucketing, gradient accumulation, cosine scheduler
- **Reliability**: split isolation, dataset SHA-256 fingerprint, optimizer/scheduler/scaler checkpointing, resume fingerprint checks

A smoke run validates execution only; do not interpret it as evidence of response quality. Full conversational evaluation is a separate gate.

## 1. Environment Setup & GPU Verification

In [1]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"bfloat16 Supported: {torch.cuda.is_bf16_supported()}")

Sun Nov  1 11:02:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Project Code & Repository Setup

In [2]:
# Mount Drive for dataset and persistent checkpoint storage.
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
project_root = '/content/Mirror-7-main'
if not os.path.isdir(os.path.join(project_root, '.git')):
    if os.path.exists(project_root):
        raise RuntimeError(f'{project_root} exists but is not a Git checkout. Move it or choose a clean path.')
    subprocess.run(['git', 'clone', 'https://github.com/realahmad07/Mirror-7.git', project_root], check=True)
sys.path.insert(0, project_root)
os.chdir(project_root)
print(f'Working Directory: {os.getcwd()}')

Mounted at /content/drive
Working Directory: /content/Mirror-7-main


## 3. Dataset Access & Fingerprint Verification

In [3]:
from pathlib import Path
import shutil
from training.dataset import load_jsonl, validate_split_isolation, summarize

dataset_path = Path('training/data/mirror_oasst2_curated_v1.jsonl')
drive_dataset = Path('/content/drive/MyDrive/Mirror7/data/mirror_oasst2_curated_v1.jsonl')
if not dataset_path.is_file():
    if not drive_dataset.is_file():
        raise FileNotFoundError(f'Dataset missing. Place it at {dataset_path} or {drive_dataset}.')
    dataset_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_dataset, dataset_path)
examples = load_jsonl(dataset_path)
validate_split_isolation(examples)
summary = summarize(examples)

print('=== DATASET INTEGRITY VERIFIED ===')
print(f'Total Examples: {summary["total"]}')
print(f'Splits: {summary["splits"]}')
print(f'Fingerprint: {summary["fingerprint"]}')

=== DATASET INTEGRITY VERIFIED ===
Total Examples: 14627
Splits: {'test': 1469, 'train': 11703, 'validation': 1455}
Fingerprint: e7d66d84725bfc934ea7b8656bfb815625f80adc81cd05d1ea868e6585c4715a


## 4. Training Configuration Selection

In [4]:
# Start conservatively on a 15 GB T4. Increase only after observing peak VRAM.
BATCH_SIZE = 16
MAX_BYTES = 1024
LEARNING_RATE = 3e-4
GRAD_ACCUM = 2
EPOCHS = 5
USE_AMP = True
OUTPUT_CHECKPOINT = '/content/drive/MyDrive/Mirror7/checkpoints/mirror7_native_v5.pt'

print(f'Config: batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, max_bytes={MAX_BYTES}, lr={LEARNING_RATE}, epochs={EPOCHS}')

Config: batch=16, grad_accum=2, max_bytes=1024, lr=0.0003, epochs=5


## 5. Quick GPU Smoke Test (Verification before full run)

In [5]:
import torch
from training.gpu_train import train_gpu

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU not detected. Select a Colab GPU runtime before continuing.')
print(f'GPU: {torch.cuda.get_device_name(0)}')

smoke_report = train_gpu(
    dataset_path=dataset_path,
    output_path='artifacts/smoke_colab_test.pt',
    epochs=1,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    max_bytes=MAX_BYTES,
    grad_accum_steps=GRAD_ACCUM,
    use_amp=USE_AMP,
    smoke_test_steps=2,
)
print('GPU smoke test completed:', smoke_report['status'])
print('Smoke checkpoint:', smoke_report['output_checkpoint'])

GPU: Tesla T4
Hardware Config: {
  "device": "cuda:0",
  "device_name": "Tesla T4",
  "total_vram_gb": 14.56,
  "cuda_capability": "7.5",
  "bf16_supported": true
}
Loading dataset: training/data/mirror_oasst2_curated_v1.jsonl
Dataset Verified. Fingerprint: e7d66d84725bfc934ea7b8656bfb815625f80adc81cd05d1ea868e6585c4715a
Total: 14627, Splits: {'test': 1469, 'train': 11703, 'validation': 1455}
Model initialized: 974,019 parameters (Architecture: V5 Enhanced [BiGRU=True, MHA=4, Gated=True])
Smoke test complete at step 2.
{"epoch": 1, "train_loss": 5.6142, "validation_loss": 5.1083, "duration_seconds": 11.36, "throughput_bytes_sec": 3245.3, "peak_vram_mb": 3640.1}

--- Generation Check ---
Prompt: Explain how cross-attention works in 2 sentences.
Response: eM\ufffd\ufffd A\ufffd\u049dN0\ufffd\ufffde\ufffda\ufffdhaA3\ufffdce .\ufffd \ufffd\ufffd\ufffd!e!\ufffd.c\ufffdAp\ufffdA\ufffd\ufffd\ufffd \ufffd\ufffd!\ufffd\ufffd/\ufffd\ufffd p.Ah\ufffd\ufffdo.\ufffdm eV\ufffd<\ufffd  \ufffdT\u

## 6. Full Training Run (or Resuming Interrupted Run)

In [6]:
# Resume only from a checkpoint produced with the exact same dataset fingerprint.
RESUME_PATH = None  # Set to OUTPUT_CHECKPOINT only when continuing the same dataset/config run.
Path(OUTPUT_CHECKPOINT).parent.mkdir(parents=True, exist_ok=True)

training_result = train_gpu(
    dataset_path=dataset_path,
    output_path=OUTPUT_CHECKPOINT,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    max_bytes=MAX_BYTES,
    grad_accum_steps=GRAD_ACCUM,
    use_amp=USE_AMP,
    resume_path=RESUME_PATH,
    smoke_test_steps=0,
)

print('Training Complete!')
print('Final Checkpoint:', training_result['output_checkpoint'])
print('Best Checkpoint:', training_result['best_checkpoint'])

Hardware Config: {
  "device": "cuda:0",
  "device_name": "Tesla T4",
  "total_vram_gb": 14.56,
  "cuda_capability": "7.5",
  "bf16_supported": true
}
Loading dataset: training/data/mirror_oasst2_curated_v1.jsonl
Dataset Verified. Fingerprint: e7d66d84725bfc934ea7b8656bfb815625f80adc81cd05d1ea868e6585c4715a
Total: 14627, Splits: {'test': 1469, 'train': 11703, 'validation': 1455}
Model initialized: 974,019 parameters (Architecture: V5 Enhanced [BiGRU=True, MHA=4, Gated=True])
{"epoch": 1, "train_loss": 2.6696, "validation_loss": 2.0974, "duration_seconds": 1978.65, "throughput_bytes_sec": 6800.3, "peak_vram_mb": 9081.3}
{"epoch": 2, "train_loss": 2.2833, "validation_loss": 1.9263, "duration_seconds": 2123.62, "throughput_bytes_sec": 6336.0, "peak_vram_mb": 9081.3}
{"epoch": 3, "train_loss": 2.1593, "validation_loss": 1.8378, "duration_seconds": 2014.92, "throughput_bytes_sec": 6677.9, "peak_vram_mb": 9081.3}
{"epoch": 4, "train_loss": 2.0979, "validation_loss": 1.786, "duration_seconds"

## 7. Generation Evaluation & Prompt Sensitivity Check

In [ ]:
from training.native_conversation_v5 import load_checkpoint

best_ckpt = training_result['best_checkpoint']
model, payload = load_checkpoint(best_ckpt, device='cuda' if torch.cuda.is_available() else 'cpu')
model.eval()

test_prompts = [
    "What is Mirror 7?",
    "Explain how cross-attention works.",
    "Calculate 15 * 12 step by step.",
    "Write a Python function to check if a number is prime.",
]

for p in test_prompts:
    output = model.generate_text(p, max_new_tokens=256, temperature=0.7)
    print(f"\nPROMPT: {p}")
    print(f"RESPONSE: {output}")
    print("-" * 50)